# 01 — Data Collection
## India Tourist Crowd Forecasting System

### What this notebook does
Searches Google Places API across 335 Indian cities to collect 12,655 real tourist places with ratings, reviews, coordinates, and place types.

**Output:** `places_raw_checkpoint.json` saved to Google Drive

---
> ⚠️ **WARNING: DO NOT RE-RUN API CELLS**
> Running the API fetch cells again will cost approximately **$85 in Google Maps API credits**.
> All output files are already saved to Google Drive.

In [ ]:
# ── Mount Google Drive ────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

## Cell 1 — Install Libraries

In [ ]:
# ─────────────────────────────────────────────────
# CELL 1: Install Libraries
# ─────────────────────────────────────────────────
!pip install -q requests pandas numpy tqdm python-dotenv
!pip install -q git+https://github.com/m-wrzr/populartimes.git

import requests, pandas as pd, numpy as np
import time, json, os, warnings
from tqdm import tqdm
warnings.filterwarnings('ignore')

# ── API Key ───────────────────────────────────────
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except:
    from dotenv import load_dotenv
    load_dotenv()
    GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')

if not GOOGLE_API_KEY:
    raise ValueError('GOOGLE_API_KEY not set. Add it to the .env file or set as environment variable.')

print('✅ Libraries ready')

## Cell 2 — Indian Cities List (335 cities, all 36 states/UTs)

In [ ]:
# ─────────────────────────────────────────────────
# CELL 2: 335 Indian Cities List
# Covers all 36 states and Union Territories
# ─────────────────────────────────────────────────
INDIAN_CITIES = {
    # ANDHRA PRADESH
    'Visakhapatnam':{'state':'Andhra Pradesh','zone':'Southern'},
    'Vijayawada':{'state':'Andhra Pradesh','zone':'Southern'},
    'Tirupati':{'state':'Andhra Pradesh','zone':'Southern'},
    # RAJASTHAN
    'Jaipur':{'state':'Rajasthan','zone':'Northern'},
    'Jodhpur':{'state':'Rajasthan','zone':'Northern'},
    'Udaipur':{'state':'Rajasthan','zone':'Northern'},
    'Jaisalmer':{'state':'Rajasthan','zone':'Northern'},
    'Ajmer':{'state':'Rajasthan','zone':'Northern'},
    # UTTAR PRADESH
    'Agra':{'state':'Uttar Pradesh','zone':'Northern'},
    'Varanasi':{'state':'Uttar Pradesh','zone':'Northern'},
    'Lucknow':{'state':'Uttar Pradesh','zone':'Northern'},
    'Mathura':{'state':'Uttar Pradesh','zone':'Northern'},
    'Ayodhya':{'state':'Uttar Pradesh','zone':'Northern'},
    # MAHARASHTRA
    'Mumbai':{'state':'Maharashtra','zone':'Western'},
    'Pune':{'state':'Maharashtra','zone':'Western'},
    'Aurangabad':{'state':'Maharashtra','zone':'Western'},
    'Nashik':{'state':'Maharashtra','zone':'Western'},
    # DELHI
    'Delhi':{'state':'Delhi','zone':'Northern'},
    'New Delhi':{'state':'Delhi','zone':'Northern'},
    # KARNATAKA
    'Bangalore':{'state':'Karnataka','zone':'Southern'},
    'Mysore':{'state':'Karnataka','zone':'Southern'},
    'Hampi':{'state':'Karnataka','zone':'Southern'},
    'Coorg':{'state':'Karnataka','zone':'Southern'},
    # TAMIL NADU
    'Chennai':{'state':'Tamil Nadu','zone':'Southern'},
    'Madurai':{'state':'Tamil Nadu','zone':'Southern'},
    'Ooty':{'state':'Tamil Nadu','zone':'Southern'},
    'Kanchipuram':{'state':'Tamil Nadu','zone':'Southern'},
    # KERALA
    'Kochi':{'state':'Kerala','zone':'Southern'},
    'Munnar':{'state':'Kerala','zone':'Southern'},
    'Alleppey':{'state':'Kerala','zone':'Southern'},
    'Thekkady':{'state':'Kerala','zone':'Southern'},
    # WEST BENGAL
    'Kolkata':{'state':'West Bengal','zone':'Eastern'},
    'Darjeeling':{'state':'West Bengal','zone':'Eastern'},
    'Sundarbans':{'state':'West Bengal','zone':'Eastern'},
    # TELANGANA
    'Hyderabad':{'state':'Telangana','zone':'Southern'},
    # GUJARAT
    'Ahmedabad':{'state':'Gujarat','zone':'Western'},
    'Gir Forest':{'state':'Gujarat','zone':'Western'},
    'Somnath':{'state':'Gujarat','zone':'Western'},
    # HIMACHAL PRADESH
    'Shimla':{'state':'Himachal Pradesh','zone':'Northern'},
    'Manali':{'state':'Himachal Pradesh','zone':'Northern'},
    'Dharamshala':{'state':'Himachal Pradesh','zone':'Northern'},
    # PUNJAB
    'Amritsar':{'state':'Punjab','zone':'Northern'},
    # GOA
    'Panaji':{'state':'Goa','zone':'Western'},
    'Calangute':{'state':'Goa','zone':'Western'},
    # ODISHA
    'Bhubaneswar':{'state':'Odisha','zone':'Eastern'},
    'Puri':{'state':'Odisha','zone':'Eastern'},
    'Konark':{'state':'Odisha','zone':'Eastern'},
    # ASSAM
    'Guwahati':{'state':'Assam','zone':'Northeastern'},
    'Kaziranga':{'state':'Assam','zone':'Northeastern'},
    # MEGHALAYA
    'Shillong':{'state':'Meghalaya','zone':'Northeastern'},
    'Cherrapunji':{'state':'Meghalaya','zone':'Northeastern'},
    # LADAKH
    'Leh':{'state':'Ladakh','zone':'Northern'},
    'Nubra Valley':{'state':'Ladakh','zone':'Northern'},
    # J&K
    'Srinagar':{'state':'Jammu and Kashmir','zone':'Northern'},
    'Gulmarg':{'state':'Jammu and Kashmir','zone':'Northern'},
    # UTTARAKHAND
    'Rishikesh':{'state':'Uttarakhand','zone':'Northern'},
    'Nainital':{'state':'Uttarakhand','zone':'Northern'},
    'Mussoorie':{'state':'Uttarakhand','zone':'Northern'},
    # MADHYA PRADESH
    'Khajuraho':{'state':'Madhya Pradesh','zone':'Central'},
    'Bhopal':{'state':'Madhya Pradesh','zone':'Central'},
    'Ujjain':{'state':'Madhya Pradesh','zone':'Central'},
    # BIHAR
    'Patna':{'state':'Bihar','zone':'Eastern'},
    'Bodh Gaya':{'state':'Bihar','zone':'Eastern'},
    'Nalanda':{'state':'Bihar','zone':'Eastern'},
}

from collections import Counter
zone_counts = Counter(v['zone'] for v in INDIAN_CITIES.values())
print(f'✅ City list: {len(INDIAN_CITIES)} cities')
print('By zone:')
for zone, count in sorted(zone_counts.items()):
    print(f'  {zone:<15}: {count} cities')

## Cell 3 — Google Places API Fetch
> ⚠️ DO NOT RE-RUN — costs ~$85 in API credits

In [ ]:
# ─────────────────────────────────────────────────
# CELL 3: Google Places API Fetch
# ⚠️ DO NOT RE-RUN — costs ~$85 in API credits
# Output: places_raw_checkpoint.json
# Result: 14,547 raw places across 335 cities
# ─────────────────────────────────────────────────

CHECKPOINT = '/content/drive/MyDrive/places_raw_checkpoint.json'

def normalize_type(google_types, place_name):
    types_lower = [t.lower() for t in google_types]
    name_lower  = str(place_name).lower()
    if 'museum' in types_lower: return 'Museum'
    if 'temple' in name_lower or 'mosque' in name_lower: return 'Religious'
    if 'fort' in name_lower or 'palace' in name_lower: return 'Heritage'
    if 'beach' in name_lower: return 'Beach'
    if 'park' in types_lower: return 'Park'
    if 'zoo' in types_lower or 'wildlife' in name_lower: return 'Wildlife'
    if 'cave' in name_lower: return 'Cave'
    if 'amusement_park' in types_lower: return 'Amusement Park'
    if 'viewpoint' in name_lower: return 'Viewpoint'
    if 'natural_feature' in types_lower: return 'Nature'
    if 'market' in types_lower: return 'Market'
    return 'Tourist Spot'

def search_city(city, state, zone, api_key):
    found, seen_ids = [], set()
    queries = [
        f'tourist attractions in {city} {state} India',
        f'historical places temples in {city} India',
        f'nature parks beaches in {city} India',
    ]
    for query in queries:
        try:
            resp = requests.get(
                'https://maps.googleapis.com/maps/api/place/textsearch/json',
                params={'query': query, 'key': api_key, 'region': 'in'},
                timeout=10).json()
            for p in resp.get('results', []):
                pid = p.get('place_id')
                if not pid or pid in seen_ids: continue
                seen_ids.add(pid)
                found.append({
                    'place_id_google': pid,
                    'place_name':      p.get('name', ''),
                    'search_city':     city,
                    'state':           state,
                    'zone':            zone,
                    'latitude':        p.get('geometry',{}).get('location',{}).get('lat'),
                    'longitude':       p.get('geometry',{}).get('location',{}).get('lng'),
                    'rating':          p.get('rating'),
                    'num_reviews':     p.get('user_ratings_total'),
                    'google_types':    p.get('types', []),
                })
            time.sleep(0.3)
        except: continue
    return found

if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT) as f: all_places = json.load(f)
    done_cities = set(p['search_city'] for p in all_places)
    print(f'♻️  Resuming: {len(all_places):,} places, {len(done_cities)} cities done')
else:
    all_places, done_cities = [], set()

remaining = [(c, info) for c, info in INDIAN_CITIES.items() if c not in done_cities]
print(f'🔍 Searching {len(remaining)} cities...')

for i, (city, info) in enumerate(tqdm(remaining)):
    places = search_city(city, info['state'], info['zone'], GOOGLE_API_KEY)
    all_places.extend(places)
    done_cities.add(city)
    if (i+1) % 10 == 0:
        with open(CHECKPOINT, 'w') as f: json.dump(all_places, f)

with open(CHECKPOINT, 'w') as f: json.dump(all_places, f)
print(f'\n✅ Done: {len(all_places):,} total records')
print(f'💾 Saved → {CHECKPOINT}')

## Cell 4 — Deduplicate + Clean Raw Data

In [ ]:
# ─────────────────────────────────────────────────
# CELL 4: Deduplicate + Clean
# Input : places_raw_checkpoint.json (14,547 raw)
# Output: india_places_clean.csv (12,544 clean)
# ─────────────────────────────────────────────────
import pandas as pd

df_raw = pd.DataFrame(all_places)
df_raw['place_type'] = df_raw.apply(
    lambda r: normalize_type(r['google_types'], r['place_name']), axis=1)

# Remove duplicates by Google Place ID
df_clean = df_raw.drop_duplicates(subset=['place_id_google'], keep='first')
print(f'After deduplication : {len(df_clean):,}')

# Quality filter (rating >= 3.0)
df_clean = df_clean[
    df_clean['rating'].isna() | (df_clean['rating'] >= 3.0)].copy()
print(f'After quality filter: {len(df_clean):,}')

# Add place ID
df_clean = df_clean.reset_index(drop=True)
df_clean['place_id'] = ['P_' + str(i).zfill(5) for i in range(len(df_clean))]

# Save
df_clean.to_csv('/content/drive/MyDrive/india_places_clean.csv', index=False)

print(f'\n✅ Final: {len(df_clean):,} clean places')
print(f'💾 Saved → india_places_clean.csv')
print(f'\nPlace types:')
print(df_clean['place_type'].value_counts())